# 📊 EDA — Hà Nội AQI 2022–2025
> Phân tích khám phá dữ liệu: phân phối AQI theo giờ, boxplot theo mùa, heatmap tương quan, trend 4 năm, tỷ lệ AQI Category

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from numpy.polynomial.polynomial import polyfit
import warnings
warnings.filterwarnings('ignore')

## CẤU HÌNH CHUNG

In [ ]:
import os as _os
# Chỉnh đường dẫn nếu cần
DATA_PATH = _os.path.join('..', 'clean', 'hanoi_aqi_cleaned.csv')
OUT       = './'   # output lưu tại thư mục hiện tại

PALETTE = {
    'Good':                           '#2ecc71',
    'Moderate':                       '#f1c40f',
    'Unhealthy for sensitive groups': '#e67e22',
    'Unhealthy':                      '#e74c3c',
    'Very Unhealthy':                 '#9b59b6',
    'Hazardous':                      '#7f1d1d',
}
CAT_ORDER    = list(PALETTE.keys())
SEASON_MAP   = {0: 'Spring', 1: 'Summer', 2: 'Autumn', 3: 'Winter'}
SEASON_ORDER = ['Spring', 'Summer', 'Autumn', 'Winter']
SEASON_CLR   = {'Spring': '#27ae60', 'Summer': '#f39c12',
                'Autumn': '#e67e22', 'Winter': '#3498db'}

sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.dpi': 150, 'axes.titlepad': 10})

## ĐỌC DỮ LIỆU

In [ ]:
df = pd.read_csv(DATA_PATH)
df['aqi_category'] = pd.Categorical(df['aqi_category'],
                                    categories=CAT_ORDER, ordered=True)
df['season_name'] = df['season'].map(SEASON_MAP)
df['ym'] = pd.to_datetime(
    df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2)
)

print(f"Dữ liệu: {df.shape[0]:,} hàng × {df.shape[1]} cột")
print("Phân bố AQI Category:\n", df['aqi_category'].value_counts(), "\n")

## EDA 1 — Phân phối AQI trung bình theo giờ trong ngày

In [ ]:
print("── EDA 1: AQI theo giờ ─────────────────────────────────────────────")
hourly = df.groupby('hour')['aqi'].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hourly.index, hourly.values,
        color='#e74c3c', lw=2.5, marker='o', ms=5, zorder=3)
ax.fill_between(hourly.index, hourly.values, alpha=0.15, color='#e74c3c')
ax.set_xticks(range(24))
ax.set_xlabel('Giờ trong ngày', fontsize=12)
ax.set_ylabel('AQI trung bình', fontsize=12)
ax.set_title('Phân phối AQI trung bình theo giờ trong ngày — Hà Nội 2022–2025',
             fontsize=13, fontweight='bold')

peak_h = int(hourly.idxmax())
best_h = int(hourly.idxmin())
ax.annotate(f'Đỉnh ô nhiễm: {hourly[peak_h]:.1f} lúc {peak_h}h',
            xy=(peak_h, hourly[peak_h]),
            xytext=(peak_h + 1.8, hourly[peak_h] + 3),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', fc='#ffe6e6', ec='#e74c3c', alpha=0.8))
ax.annotate(f'Sạch nhất: {hourly[best_h]:.1f} lúc {best_h}h',
            xy=(best_h, hourly[best_h]),
            xytext=(best_h + 1.8, hourly[best_h] - 7),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', fc='#e8f8f5', ec='#27ae60', alpha=0.8))
plt.tight_layout()
plt.show()
print(f"  → Giờ ô nhiễm nhất: {peak_h}h  (AQI={hourly[peak_h]:.1f})")
print(f"  → Giờ sạch nhất:    {best_h}h  (AQI={hourly[best_h]:.1f})")
print()

## EDA 2 — Boxplot AQI theo 4 mùa

In [ ]:
print("── EDA 2: Boxplot theo mùa ──────────────────────────────────────────")
season_stats = df.groupby('season_name')['aqi'].agg(['mean', 'median'])
worst_season = season_stats['mean'].idxmax()

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=df, x='season_name', y='aqi', order=SEASON_ORDER,
            palette=SEASON_CLR, width=0.5, linewidth=1.5,
            flierprops=dict(marker='o', markersize=2, alpha=0.3), ax=ax)
ax.set_xlabel('Mùa', fontsize=12)
ax.set_ylabel('AQI', fontsize=12)
ax.set_title('Phân phối AQI theo 4 mùa — Hà Nội 2022–2025',
             fontsize=13, fontweight='bold')

med = season_stats.loc[worst_season, 'median']
idx = SEASON_ORDER.index(worst_season)
ax.annotate(f'AQI cao nhất:\n{worst_season} (median={med:.0f})',
            xy=(idx, med),
            xytext=(idx + 0.4, med + 20),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='#fdecea', ec='#e74c3c', alpha=0.9))

for i, s in enumerate(SEASON_ORDER):
    m = season_stats.loc[s, 'mean']
    ax.text(i, ax.get_ylim()[0] + 5, f'avg={m:.0f}',
            ha='center', fontsize=8.5, color='#333')
plt.tight_layout()
plt.show()
print(f"  → Mùa AQI cao nhất: {worst_season}")
for s in SEASON_ORDER:
    print(f"     {s}: mean={season_stats.loc[s,'mean']:.1f}  median={season_stats.loc[s,'median']:.1f}")
print()

## EDA 3 — Heatmap tương quan 13 features

In [ ]:
print("── EDA 3: Correlation Heatmap ───────────────────────────────────────")
FEAT13 = ['aqi', 'co', 'no2', 'o3', 'pm10', 'pm25', 'so2',
          'clouds', 'precipitation', 'pressure', 'relative_humidity',
          'temperature', 'wind_speed']
corr = df[FEAT13].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, linewidths=0.5, annot_kws={'size': 8},
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Heatmap tương quan 13 features — Hà Nội AQI',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# In top correlations với AQI
aqi_corr = corr['aqi'].drop('aqi').abs().sort_values(ascending=False)
print("  → Top 5 features tương quan với AQI:")
for feat, val in aqi_corr.head(5).items():
    print(f"     {feat}: {corr['aqi'][feat]:+.3f}")
print()

## EDA 4 — Trend AQI trung bình theo tháng 2022–2025

In [ ]:
print("── EDA 4: Trend theo tháng ──────────────────────────────────────────")
monthly = df.groupby('ym')['aqi'].mean().reset_index()
x_num   = np.arange(len(monthly))
c       = polyfit(x_num, monthly['aqi'].values, 1)
trend   = c[0] + c[1] * x_num
direction = '↑ TĂNG' if c[1] > 0 else '↓ GIẢM'

fig, ax = plt.subplots(figsize=(14, 5))
# Nền từng năm
for yr, col in zip([2022, 2023, 2024, 2025],
                   ['#fdfefe', '#eaf4fb', '#fdfefe', '#fef9e7']):
    ax.axvspan(pd.Timestamp(f'{yr}-01-01'),
               pd.Timestamp(f'{yr}-12-31'),
               alpha=0.35, color=col, lw=0)
    ax.text(pd.Timestamp(f'{yr}-07-01'), ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 60,
            str(yr), ha='center', fontsize=9, color='#aaa')

ax.plot(monthly['ym'], monthly['aqi'],
        color='#2980b9', lw=2, marker='o', ms=4, zorder=3, label='AQI tháng')
ax.fill_between(monthly['ym'], monthly['aqi'], alpha=0.12, color='#2980b9')
ax.plot(monthly['ym'], trend, '--', color='#e74c3c', lw=2,
        label=f'Xu hướng tổng thể ({direction}, slope={c[1]:.2f}/tháng)')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
plt.xticks(rotation=45, ha='right')
ax.set_xlabel('Tháng', fontsize=12)
ax.set_ylabel('AQI trung bình', fontsize=12)
ax.set_title('Trend AQI trung bình theo tháng — Hà Nội 2022–2025',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

yearly = df.groupby('year')['aqi'].mean()
print(f"  → Xu hướng: {direction}  (slope={c[1]:.4f}/tháng)")
print("  → AQI trung bình từng năm:")
for yr, v in yearly.items():
    print(f"     {yr}: {v:.1f}")
print()

## EDA 5 — Tỷ lệ 6 mức AQI Category

In [ ]:
print("── EDA 5: Tỷ lệ AQI Category ────────────────────────────────────────")
cat_counts = df['aqi_category'].value_counts().reindex(CAT_ORDER)
colors     = [PALETTE[c] for c in CAT_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Pie chart
wedges, _, autotexts = axes[0].pie(
    cat_counts, labels=None, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5),
    pctdistance=0.78)
for at in autotexts:
    at.set_fontsize(9)
axes[0].legend(wedges,
               [f'{c}  ({v:,})' for c, v in zip(CAT_ORDER, cat_counts)],
               loc='lower center', bbox_to_anchor=(0.5, -0.15),
               fontsize=8, ncol=2)
axes[0].set_title('Tỷ lệ 6 mức AQI Category', fontsize=12, fontweight='bold')

# Bar chart
bars = axes[1].bar(range(len(CAT_ORDER)), cat_counts.values,
                   color=colors, edgecolor='white', linewidth=0.8)
axes[1].set_xticks(range(len(CAT_ORDER)))
axes[1].set_xticklabels(
    [c.replace(' for sensitive groups', '\nfor sensitive groups')
     for c in CAT_ORDER], fontsize=8.5)
axes[1].set_ylabel('Số bản ghi', fontsize=11)
axes[1].set_title('Số lượng theo 6 mức AQI', fontsize=12, fontweight='bold')
for bar, v in zip(bars, cat_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 80,
                 f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

print("  → Phân bố AQI Category:")
for cat, v in cat_counts.items():
    print(f"     {cat:<38}: {v:>6,}  ({v/len(df)*100:.1f}%)")
print()

## NHẬN XÉT PHÂN TÍCH TỔNG HỢP

In [ ]:
print("=" * 65)
print("  NHẬN XÉT PHÂN TÍCH EDA")
print("=" * 65)
print(f"  1. Giờ ô nhiễm nhất : {peak_h}h (AQI={hourly[peak_h]:.1f}) — ban đêm")
print(f"     Giờ sạch nhất     : {best_h}h (AQI={hourly[best_h]:.1f}) — chiều")
print(f"  2. Mùa AQI cao nhất : {worst_season} (avg AQI={season_stats.loc[worst_season,'mean']:.1f})")
print(f"  3. Xu hướng 4 năm   : {direction} (slope={c[1]:.4f}/tháng)")
print(f"     2022→{yearly.index[-1]}: {yearly.iloc[0]:.1f} → {yearly.iloc[-1]:.1f}")
print(f"  4. Class mất cân bằng: Good {cat_counts['Good']/len(df)*100:.1f}%,"
      f" Hazardous {cat_counts['Hazardous']/len(df)*100:.1f}%")
print("     → Áp dụng SMOTE để cân bằng trước khi train")
print("=" * 65)
print()